In [ ]:
import os

In [2]:
os.listdir('./')

['.ipynb_checkpoints',
 'accountant_2015_2024_original.xlsx',
 'accountant_trx_02_04_2025_new.xlsx',
 'Untitled.ipynb']

In [3]:
import pandas as pd

In [4]:
accountant_trx_org = pd.read_excel('./accountant_2015_2024_original.xlsx', sheet_name='Sheet1')

accountant_trx_org['APP_SOURCE_CODE'] = accountant_trx_org['APP_SOURCE_CODE'].str.strip()
accountant_trx_org['TRANSACTION_CODE_ATB'] = accountant_trx_org['TRANSACTION_CODE_ATB'].astype(int).astype(str).str.zfill(3)

accountant_trx_org.head()

,APP_SOURCE_CODE,TRANSACTION_CODE_ATB,TRANSACTION_DESC_ATB
0,T24TL,297,Comm.OPPOS EFFET
1,UAJL,920,Interets
2,@@CC,120,Remb avance placement
3,TLVIR,283,Commission virement re\u
4,@@FM,401,Exchange Delivery


In [5]:
# Drop duplicates based on specified columns, keeping the first occurrence
#df.drop_duplicates(subset=['APP_SOURCE_CODE', 'TRANSACTION_CODE_ATB', 'TRANSACTION_DESC_ATB'], keep='first', inplace=True)
accountant_trx_org_nd = accountant_trx_org.drop_duplicates(subset=['APP_SOURCE_CODE', 'TRANSACTION_CODE_ATB', 'TRANSACTION_DESC_ATB'], keep='first')

In [6]:
accountant_trx_org_nd

,APP_SOURCE_CODE,TRANSACTION_CODE_ATB,TRANSACTION_DESC_ATB
0,T24TL,297,Comm.OPPOS EFFET
1,UAJL,920,Interets
2,@@CC,120,Remb avance placement
3,TLVIR,283,Commission virement re\u
4,@@FM,401,Exchange Delivery
...,...,...,...
12245,TCTL3,223,Paiement effet
12246,TCTL0,510,Credit
12247,UDHD,878,Transf Sld Data cleaning
12250,USR6,010,Debit


In [9]:
accountant_trx_new = pd.read_excel('./accountant_trx_02_04_2025_new.xlsx', sheet_name='trx_code')

accountant_trx_new['APP_SOURCE_CODE'] = accountant_trx_new['APP_SOURCE_CODE'].str.strip()
accountant_trx_new['TRANSACTION_CODE_ATB'] = accountant_trx_new['TRANSACTION_CODE_ATB'].astype(int).astype(str).str.zfill(3)

accountant_trx_new.head()

,TRANSACTION_CODE_ATB,TRANSACTION_DESC_ATB,APP_SOURCE_CODE,APP_SOURCE,OP_TYPE,Categorie_de_transaction,TRANSACTION_CODE_GOAML,TRANSACTION_DESC_GOAML,FUND_CODE_1,FUND_CODE_2
0,001,Retrait Especes,TCTL4,NaN,Caisse,Le code affectant directement le compte client...,B200,Retrait espèces,A,K
1,002,Mise a Disposition Cash,UYBO,NaN,Caisse,Le code affectant directement le compte client...,B372,Mise à disposition,A,K
2,005,Paiement Honoraires,MB073,NaN,NaN,a clarifier,NaN,NaN,NaN,NaN
3,006,Annulation Reglement Honoraires,MB071,NaN,NaN,a clarifier,NaN,NaN,NaN,NaN
4,007,Mise a Disposition,TCTL4,NaN,Caisse,Le code affectant directement le compte client...,B372,Mise à disposition,A,K


In [16]:
accountant_trx_f = pd.merge(
    left=accountant_trx_org_nd,
    right=accountant_trx_new,
    on=['APP_SOURCE_CODE', 'TRANSACTION_CODE_ATB'],
    how='left',
    suffixes=('_ACC', '_F'),
    indicator=True
)

In [21]:
accountant_trx_f

,APP_SOURCE_CODE,TRANSACTION_CODE_ATB,TRANSACTION_DESC_ATB_ACC,TRANSACTION_DESC_ATB_F,APP_SOURCE,OP_TYPE,Categorie_de_transaction,TRANSACTION_CODE_GOAML,TRANSACTION_DESC_GOAML,FUND_CODE_1,FUND_CODE_2,_merge
0,T24TL,297,Comm.OPPOS EFFET,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
1,UAJL,920,Interets,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
2,@@CC,120,Remb avance placement,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3,TLVIR,283,Commission virement re\u,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
4,@@FM,401,Exchange Delivery,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
...,...,...,...,...,...,...,...,...,...,...,...,...
3762,TCTL3,223,Paiement effet,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3763,TCTL0,510,Credit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only
3764,UDHD,878,Transf Sld Data cleaning,Transf Sld Data cleaning,NaN,NaN,NaN,NaN,NaN,NaN,NaN,both
3765,USR6,010,Debit,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,left_only


In [20]:
accountant_trx_f.to_excel('accountant_trx_f.xlsx', sheet_name='Sheet1', index=False)

In [29]:
accountant_trx_f.query("OP_TYPE == 'Caisse' and Categorie_de_transaction == 'code affectant directement le compte client (débit/crédit)'")

,APP_SOURCE_CODE,TRANSACTION_CODE_ATB,TRANSACTION_DESC_ATB_ACC,TRANSACTION_DESC_ATB_F,APP_SOURCE,OP_TYPE,Categorie_de_transaction,TRANSACTION_CODE_GOAML,TRANSACTION_DESC_GOAML,FUND_CODE_1,FUND_CODE_2,_merge
15,T24TL,162,Achat BBE Doss Scolarite,Achat BBE Doss Scolarit{,Teller,Caisse,code affectant directement le compte client (d...,NaN,NaN,NaN,NaN,both
38,T24TL,086,Retrocession de BBE Contre TND,Retrocession de BBE Contre TND,Teller,Caisse,code affectant directement le compte client (d...,B301,achats de devises,K,K,both
46,T24TL,179,Ret Chq meme AG,Ret Chq meme AG,Teller,Caisse,code affectant directement le compte client (d...,B123,NaN,A,K,both
103,T24TL,127,Emission Mise a Disposition,Emission Mise a Disposition,Teller,Caisse,code affectant directement le compte client (d...,NaN,NaN,NaN,NaN,both
118,T24TL,171,Vers especes BQE a Domicile,Vers especes BQE a Domicile,Teller,Caisse,code affectant directement le compte client (d...,B114,Versement espèces,K,A,both
153,T24TL,655,Retrocession BBE AVA,Retrocession BBE AVA,Teller,Caisse,code affectant directement le compte client (d...,B405,Annulation opération,K,K,both
219,T24TL,146,Achat BBE par CR de cpte TND,Achat BBE par CR de cpte TND,Teller,Caisse,code affectant directement le compte client (d...,B301,achats de devises,K,A,both
327,T24TL,671,Vers especes BQE a Domicile,Vers especes BQE a Domicile,Teller,Caisse,code affectant directement le compte client (d...,B114,Versement espèces,K,A,both
339,T24TL,647,Vente BBE par CR cpte ope exp,Vente BBE par CR cpte ope exp,Teller,Caisse,code affectant directement le compte client (d...,B403,Ventes de devises,K,A,both
340,T24TL,644,Vers TND sur cpt TNC meme ag,Vers TND sur cpt TNC meme ag,Teller,Caisse,code affectant directement le compte client (d...,B114,Versement espèces,K,A,both


In [25]:
accountant_trx_f[accountant_trx_f['Categorie_de_transaction'] == 'code affectant directement le compte client (débit/crédit)']

,APP_SOURCE_CODE,TRANSACTION_CODE_ATB,TRANSACTION_DESC_ATB_ACC,TRANSACTION_DESC_ATB_F,APP_SOURCE,OP_TYPE,Categorie_de_transaction,TRANSACTION_CODE_GOAML,TRANSACTION_DESC_GOAML,FUND_CODE_1,FUND_CODE_2,_merge
15,T24TL,162,Achat BBE Doss Scolarite,Achat BBE Doss Scolarit{,Teller,Caisse,code affectant directement le compte client (d...,NaN,NaN,NaN,NaN,both
38,T24TL,086,Retrocession de BBE Contre TND,Retrocession de BBE Contre TND,Teller,Caisse,code affectant directement le compte client (d...,B301,achats de devises,K,K,both
46,T24TL,179,Ret Chq meme AG,Ret Chq meme AG,Teller,Caisse,code affectant directement le compte client (d...,B123,NaN,A,K,both
103,T24TL,127,Emission Mise a Disposition,Emission Mise a Disposition,Teller,Caisse,code affectant directement le compte client (d...,NaN,NaN,NaN,NaN,both
118,T24TL,171,Vers especes BQE a Domicile,Vers especes BQE a Domicile,Teller,Caisse,code affectant directement le compte client (d...,B114,Versement espèces,K,A,both
...,...,...,...,...,...,...,...,...,...,...,...,...
3710,T24TL,665,Vente BBE alloc tour contr TND,Vente BBE alloc tour contr TND,Teller,Caisse,code affectant directement le compte client (d...,B403,Ventes de devises,K,K,both
3723,TCTL4,017,Paiement Cheque Preavise,Paiement Cheque Preavise,NaN,Cheque,code affectant directement le compte client (d...,NaN,NaN,NaN,NaN,both
3725,T24TL,143,Achat BBE fiche Info DB Compte,Achat BBE fiche Info DB Compte,Teller,Caisse,code affectant directement le compte client (d...,B301,achats de devises,K,A,both
3730,T24TL,588,Vente Alloc Pelerinage DB Compte,Vente Alloc Pelerinage DB Compte,Teller,Caisse,code affectant directement le compte client (d...,B403,Ventes de devises,A,K,both
